# SABR Paper Reproduction Walkthrough

This notebook organizes the current project into a single step-by-step workflow so the paper replication can be run one cell at a time.

Paper:
- Choi, Hu, Kwok, *Efficient and accurate simulation of the stochastic-alpha-beta-rho model*

Project scope:
- core SABR Monte Carlo simulator
- sanity checks for limiting cases and martingale behavior
- paper-style tables and figure datasets
- quick and paper-scale validation hooks


## Contents

1. [Environment and Imports](#1.-Environment-and-Imports)
2. [Project API Overview](#2.-Project-API-Overview)
3. [Table 3 Cases](#3.-Table-3-Cases)
4. [Sanity Checks](#4.-Sanity-Checks)
5. [Paper Table 1](#5.-Paper-Table-1)
6. [Paper Table 2](#6.-Paper-Table-2)
7. [Paper Figure 1 Dataset](#7.-Paper-Figure-1-Dataset)
8. [Paper Table 4](#8.-Paper-Table-4)
9. [Paper Table 5](#9.-Paper-Table-5)
10. [Paper Table 6](#10.-Paper-Table-6)
11. [Paper Table 7 / Figure 2](#11.-Paper-Table-7-/-Figure-2)
12. [Paper Figure 3](#12.-Paper-Figure-3)
13. [Validation Summary](#13.-Validation-Summary)
14. [Optional CSV Export Helpers](#14.-Optional-CSV-Export-Helpers)


## 1. Environment and Imports

Run this first. It checks the working directory, imports the project module, and shows package versions when available.


In [ ]:
from pathlib import Path
import sys
import platform
import importlib

import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Python:', sys.version)
print('Platform:', platform.platform())
print('Project root:', PROJECT_ROOT)

for pkg in ['numpy', 'pandas', 'scipy', 'pyfeng', 'pytest']:
    mod = importlib.import_module(pkg)
    print(f'{pkg}:', getattr(mod, '__version__', 'version unavailable'))


In [ ]:
from sabr_replicate import (
    FDMConfig,
    MonteCarloConfig,
    SABRParams,
    case_table_3,
    conditional_integrated_variance_moments,
    european_call_price,
    fdm_benchmark_prices,
    figure1_moment_comparison,
    figure2_runtime_tradeoff,
    finite_difference_call_price,
    martingale_test,
    raw_moments_to_central_stats,
    run_figure3_experiment,
    run_full_validation,
    run_table1_experiment,
    run_table2_experiment,
    run_table4_experiment,
    run_table5_experiment,
    run_table6_experiment,
    run_table7_experiment,
    sample_conditional_integrated_variance,
    simulate_terminal_forward,
    simulate_terminal_forward_islah,
)
import pyfeng as pf


## 2. Project API Overview

This section is only for orientation. It lists the main functions used later in the notebook.


In [ ]:
api_overview = pd.DataFrame(
    [
        ('simulate_terminal_forward', 'Main SABR Monte Carlo terminal simulator using the paper scheme'),
        ('simulate_terminal_forward_islah', 'Appendix B / Islah-style comparison branch'),
        ('sample_conditional_integrated_variance', 'Algorithm 1 style conditional average-variance sampling'),
        ('conditional_integrated_variance_moments', 'Conditional raw moments of normalized average variance'),
        ('run_table1_experiment', 'Paper Table 1 dataset'),
        ('run_table2_experiment', 'Paper Table 2 dataset'),
        ('run_table4_experiment', 'Paper Table 4 dataset'),
        ('run_table5_experiment', 'Paper Table 5 dataset'),
        ('run_table6_experiment', 'Paper Table 6 dataset'),
        ('run_table7_experiment', 'Paper Table 7 / Figure 2 dataset'),
        ('run_figure3_experiment', 'Paper Figure 3 dataset'),
        ('run_full_validation', 'Repository validation harness'),
    ],
    columns=['function', 'purpose'],
)
api_overview


## 3. Table 3 Cases

These are the paper parameter presets used throughout later sections.


In [ ]:
case_df = pd.DataFrame(case_table_3()).T
case_df.index.name = 'case'
case_df


## 4. Sanity Checks

These cells verify model-level behavior that should hold independently of the paper tables.


### 4.1 `nu = 0` should reduce SABR to CEV


In [ ]:
params = SABRParams(f0=1.0, sigma0=0.25, nu=0.0, rho=-0.4, beta=0.3)
mc = MonteCarloConfig(maturity=1.0, step=1.0, n_paths=200_000, seed=12345)
strikes = np.array([0.6, 1.0, 1.4])

terminal = simulate_terminal_forward(params, mc)
mc_prices = np.array([european_call_price(terminal, k) for k in strikes])
cev_prices = pf.Cev(sigma=params.sigma0, beta=params.beta, is_fwd=True).price(strikes, params.f0, mc.maturity)

pd.DataFrame({
    'strike': strikes,
    'mc_price': mc_prices,
    'cev_price': cev_prices,
    'error': mc_prices - cev_prices,
})


### 4.2 `beta = 1, nu = 0` should reduce SABR to Black-Scholes / lognormal


In [ ]:
params = SABRParams(f0=1.0, sigma0=0.2, nu=0.0, rho=-0.75, beta=1.0)
mc = MonteCarloConfig(maturity=1.0, step=1.0, n_paths=200_000, seed=12345)
strikes = np.array([0.8, 1.0, 1.2])

terminal = simulate_terminal_forward(params, mc)
mc_prices = np.array([european_call_price(terminal, k) for k in strikes])
bsm_prices = pf.Bsm(sigma=params.sigma0, is_fwd=True).price(strikes, params.f0, mc.maturity)

pd.DataFrame({
    'strike': strikes,
    'mc_price': mc_prices,
    'bsm_price': bsm_prices,
    'error': mc_prices - bsm_prices,
})


### 4.3 Conditional average-variance moment spot check

This is a small diagnostic cell for the `I_t^h` machinery used inside the full simulator.


In [ ]:
sigma_t = np.array([0.2])
sigma_next = np.array([0.24])
nu = 0.4
h = 1.0

mu1, mu2_raw, mu3_raw, mu4_raw = conditional_integrated_variance_moments(sigma_t, sigma_next, nu, h)
mean, var, std, cv, skewness, ex_kurtosis = raw_moments_to_central_stats(mu1, mu2_raw, mu3_raw, mu4_raw)

pd.DataFrame({
    'mu1': mu1,
    'mu2_raw': mu2_raw,
    'mu3_raw': mu3_raw,
    'mu4_raw': mu4_raw,
    'var': var,
    'std': std,
    'cv': cv,
    'skewness': skewness,
    'ex_kurtosis': ex_kurtosis,
})


### 4.4 Martingale sanity check


In [ ]:
case_v = case_table_3()['Case V']
params = SABRParams(
    f0=case_v['f0'],
    sigma0=case_v['sigma0'],
    nu=case_v['nu'],
    rho=case_v['rho'],
    beta=case_v['beta'],
)

martingale_test(params, maturities=[1, 2, 4, 6, 8, 10], step=1.0, n_paths=30_000, seed0=101)


### 4.5 `|rho| = 1` Islah edge case stability


In [ ]:
rows = []
for beta in (0.4, 0.6, 0.8):
    params = SABRParams(f0=1.0, sigma0=0.2, nu=0.2, rho=1.0, beta=beta)
    mc = MonteCarloConfig(maturity=1.0, step=1.0, n_paths=5_000, seed=123)
    terminal = simulate_terminal_forward_islah(params, mc)
    rows.append({
        'beta': beta,
        'all_finite': bool(np.isfinite(terminal).all()),
        'all_nonnegative': bool((terminal >= 0.0).all()),
        'mean_terminal': float(np.mean(terminal)),
        'min_terminal': float(np.min(terminal)),
    })

pd.DataFrame(rows)


## 5. Paper Table 1

By default, this uses the paper table benchmarks embedded in the repo. To compare against the repo PDE benchmark instead, run the later FDM cells or pass benchmark providers in standalone scripts.


In [ ]:
table1_df = run_table1_experiment(n_paths=100_000, n_repeats=50, seed0=12345)
table1_df


## 6. Paper Table 2


In [ ]:
table2_df = run_table2_experiment(n_paths=100_000, n_repeats=50, seed0=12345)
table2_df


## 7. Paper Figure 1 Dataset

This generates the skewness and ex-kurtosis comparison data used for the figure. Plotting is optional; the dataset itself is enough for validation.


In [ ]:
figure1_df = figure1_moment_comparison(hat_nu=0.4)
figure1_df.head(12)


In [ ]:
ax = figure1_df.plot(x='z_hat', y=['exact_skewness', 'ln_skewness', 'sln_fixed_skewness', 'sln_exact_skewness'], figsize=(10, 4), title='Figure 1 style skewness comparison')
ax.set_ylabel('skewness')


## 8. Paper Table 4

Case I. This includes the simulated rows and the available analytic comparison rows.


In [ ]:
table4_df = run_table4_experiment(n_paths=100_000, n_repeats=50, seed0=12345)
table4_df


## 9. Paper Table 5

Case II.


In [ ]:
table5_df = run_table5_experiment(n_paths=100_000, n_repeats=50, seed0=12345)
table5_df


## 10. Paper Table 6

Case III. This is the runtime / bias comparison against paper-reference baseline rows.


In [ ]:
table6_df = run_table6_experiment(n_paths=100_000, n_repeats=50, seed0=12345)
table6_df


## 11. Paper Table 7 / Figure 2

This section can be slow. It reproduces the convergence and runtime trade-off dataset.


In [ ]:
table7_df = run_table7_experiment(
    n_paths_base=160_000,
    n_repeats=5,
    seed0=12345,
    benchmark_source='mc',
)
table7_df


In [ ]:
figure2_df = figure2_runtime_tradeoff(
    n_paths_base=160_000,
    n_repeats=5,
    seed0=12345,
    benchmark_source='mc',
)
figure2_df


## 12. Paper Figure 3

This compares the paper scheme against the Islah approximation across maturities.


In [ ]:
figure3_df = run_figure3_experiment(
    n_paths=100_000,
    n_repeats=2,
    seed0=12345,
    benchmark_source='mc',
)
figure3_df.head(20)


In [ ]:
pivot_forward = figure3_df.pivot_table(index='maturity', columns=['model', 'step'], values='forward_error')
pivot_option = figure3_df.pivot_table(index='maturity', columns=['model', 'step'], values='option_error')

display(pivot_forward)
display(pivot_option)


## 13. Validation Summary

Start with quick mode. The paper-scale run is much slower.


In [ ]:
validation_quick = run_full_validation(quick_mode=True)
pd.Series({
    'table1_status': validation_quick['table1_status'],
    'table2_status': validation_quick['table2_status'],
    'overall_conclusion': validation_quick['overall_conclusion'],
    'replication_conclusion': validation_quick['replication_conclusion'],
})


In [ ]:
# Uncomment this only when you want the full validation sweep.
# validation_full = run_full_validation(quick_mode=False)
# pd.Series({
#     'table1_status': validation_full['table1_status'],
#     'table2_status': validation_full['table2_status'],
#     'overall_conclusion': validation_full['overall_conclusion'],
#     'replication_conclusion': validation_full['replication_conclusion'],
# })


## 14. Optional CSV Export Helpers

Use these after running the sections you care about.


In [ ]:
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'notebook_exports'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR


In [ ]:
# Example exports. Uncomment what you need.
# table1_df.to_csv(OUTPUT_DIR / 'table1.csv', index=False)
# table2_df.to_csv(OUTPUT_DIR / 'table2.csv', index=False)
# table4_df.to_csv(OUTPUT_DIR / 'table4.csv', index=False)
# table5_df.to_csv(OUTPUT_DIR / 'table5.csv', index=False)
# table6_df.to_csv(OUTPUT_DIR / 'table6.csv', index=False)
# table7_df.to_csv(OUTPUT_DIR / 'table7.csv', index=False)
# figure1_df.to_csv(OUTPUT_DIR / 'figure1_dataset.csv', index=False)
# figure3_df.to_csv(OUTPUT_DIR / 'figure3_dataset.csv', index=False)
